### Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [3]:
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

from langchain.chat_models import init_chat_model

model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x1139415b0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x113908140>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The rating of the movie")

In [7]:
model_with_structured_output = model.with_structured_output(Movie)
model_with_structured_output

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x1139415b0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x113908140>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The rating of the movie', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'

In [10]:
# Default output
model.invoke("Provides details about the movie The lord of rings")

AIMessage(content='<think>\nOkay, the user is asking for details about "The Lord of the Rings" movie. Let me start by recalling what I know. I think there are two main versions: the 2001-2003 trilogy by Peter Jackson and the 2014-2019 series by Amazon Prime, right? The user might be referring to either, but the more famous one is definitely Jackson\'s. I should clarify that.\n\nFirst, I need to break down the original trilogy. The books were by J.R.R. Tolkien, and the movies were directed by Peter Jackson. The trilogy includes The Fellowship of the Ring, The Two Towers, and The Return of the King. Each movie was released in 2001, 2002, and 2003. They won a bunch of Oscars, especially the last one, which won 11, I think.\n\nI should mention the main plot: the Fellowship\'s quest to destroy the One Ring in the fires of Mount Doom. Key characters are Frodo, Sam, Gandalf, Aragorn, Legolas, Gimli, etc. The movies were known for their epic scale, visual effects, and attention to source mater

In [14]:
# Output using structured output with Pydantic
response = model_with_structured_output.invoke("Provides details about the movie The lord of rings")
response

Movie(title='The Lord of Rings', year=2001, director='Peter Jackson', rating=8.8)

### Message output alongside parsed structured

In [15]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The rating of the movie")

model_with_structured_output = model.with_structured_output(Movie, include_raw=True)

response = model_with_structured_output.invoke("Provide details about the movie The lord of the rings")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie "The Lord of the Rings." Let me check the tools provided. There\'s a Movie function that requires title, year, director, and rating. I need to figure out the correct information for these parameters.\n\nFirst, the title is given as "The Lord of the Rings." I should confirm the exact title. The original movie is "The Lord of the Rings: The Fellowship of the Ring," released in 2001. The director is Peter Jackson. The rating might be from IMDb; the first movie has an 8.8 rating. So, the year is 2001. Let me make sure all required fields are included. Title, year, director, and rating are all required. I\'ll structure the tool call with these details.\n', 'tool_calls': [{'id': 'f9pmf1v3x', 'function': {'arguments': '{"director":"Peter Jackson","rating":8.8,"title":"The Lord of the Rings: The Fellowship of the Ring","year":2001}', 'name': 'Movie'}, 'type': 'function'}]

### Nested Structured Output

In [19]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genre: list[str]
    budged: float | None = Field(None, description="Budget in millions USD")


model_with_structured_output = model.with_structured_output(MovieDetails)

response = model_with_structured_output.invoke("Provide details about the movie The lord of the rings")
response
    

MovieDetails(title='The Lord of the Rings', year=2001, cast=[Actor(name='Elijah Wood', role='Frodo Baggins'), Actor(name='Ian McKellen', role='Gandalf'), Actor(name='Orlando Bloom', role='Legolas Greenleaf'), Actor(name='Viggo Mortensen', role='Aragorn')], genre=['Fantasy', 'Adventure'], budged=93.0)